In [1]:
from src.model import LlamaForEmbeddingLM
from src.utils import batch_inference
from transformers import AutoTokenizer
from peft import PeftModel

import torch
from tqdm import tqdm
import pickle

In [2]:
backbone_model_path = "initial_elm_model"
peft_model_id = "5tasks_full_tuning_lora_outputs/checkpoint-37780"

# load tokenizer & elm
print(f"Loading backbone model from {backbone_model_path}")
tokenizer = AutoTokenizer.from_pretrained(backbone_model_path)

elm = LlamaForEmbeddingLM.from_pretrained(
    backbone_model_path, 
    torch_dtype=torch.bfloat16,
    device_map="cuda")

print(f"Loading PEFT model from {peft_model_id}")
lora_elm = PeftModel.from_pretrained(elm, peft_model_id)
lora_elm = lora_elm.merge_and_unload()

Loading backbone model from initial_elm_model


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Loading PEFT model from 5tasks_full_tuning_lora_outputs/checkpoint-37780


# 2.5K Real Test Embedding 

In [3]:
# load testing data
test_data_path = "../data/pubmed_rct/processed_dataset/test_with_embeddings.pkl"
print(f"Loading test data from {test_data_path}")
with open(test_data_path, 'rb') as f:
    test_dict = pickle.load(f)
test_abstracts = test_dict["abstracts"]
test_embeddings = test_dict["embeddings"]

Loading test data from ../data/pubmed_rct/processed_dataset/test_with_embeddings.pkl


In [9]:
batch_size = 8
number_of_cases = len(test_embeddings)
decoded_abstracts = []
pbar = tqdm(total=number_of_cases)
for i in range(0, number_of_cases, batch_size):
    testembs = test_embeddings[i:i+batch_size] # abstract embedding for input
    decoded_outputs = batch_inference(lora_elm, tokenizer, testembs, "cuda", task="abstract", repetition_penalty=1.2)
    decoded_abstracts += decoded_outputs
    pbar.update(len(decoded_outputs))
pbar.close()

100%|██████████| 2500/2500 [37:00<00:00,  1.13it/s]


In [12]:
# Serialize and save to file
with open('../data/pubmed_rct/decoded_abstracts/5tasks_full_tuning_decoded_abstracts.pkl', 'wb') as file:
    print(len(decoded_abstracts))
    pickle.dump(decoded_abstracts, file)

2500


# 2.5K Interpolated Test Embedding

In [13]:
interpolated_data_path = "../data/pubmed_rct/processed_dataset/interpolated_embeddings.pkl"
print(f"Loading interpolated data from {interpolated_data_path}")
with open(interpolated_data_path, 'rb') as f:
    interpolated_dict = pickle.load(f)
interpolated_embeddings = interpolated_dict["embeddings"]
print(len(interpolated_embeddings))

Loading interpolated data from ../data/pubmed_rct/processed_dataset/interpolated_embeddings.pkl
2500


In [14]:
batch_size = 8
number_of_cases = len(interpolated_embeddings)
interpolated_decoded_abstracts = []
pbar = tqdm(total=number_of_cases)
for i in range(0, number_of_cases, batch_size):
    testembs = interpolated_embeddings[i:i+batch_size] # abstract embedding for input
    interpolated_decoded_outputs = batch_inference(lora_elm, tokenizer, testembs, "cuda", task="abstract", repetition_penalty=1.2)
    interpolated_decoded_abstracts += interpolated_decoded_outputs
    pbar.update(len(interpolated_decoded_outputs))
pbar.close()

100%|██████████| 2500/2500 [35:06<00:00,  1.19it/s]


In [15]:
# Serialize and save to file
with open('../data/pubmed_rct/decoded_abstracts/5tasks_full_tuning_interpolated_decoded_abstracts.pkl', 'wb') as file:
    print(len(interpolated_decoded_abstracts))
    pickle.dump(interpolated_decoded_abstracts, file)

2500
